1.不含主元的 Doolittle 分解，返回 (L, U)。
      L：单位下三角（对角线全是 1，上三角部分全是 0）
      U：上三角
      满足 L @ U ≈ A

In [3]:
import numpy as np
def lu_decompose(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    U = np.array(A, dtype=float)         
    L = np.zeros((n, n), dtype=float)
    for k in range(n - 1):
        m = U[k + 1:, k] / U[k, k]       
        L[k + 1:, k] = m                  
        U[k + 1:, k:] -= np.outer(m, U[k, k:])
        U[k + 1:, k] = 0.0               
    L += np.eye(n)                    
    U = np.triu(U)
    return L, U
if __name__ == "__main__":  #是主角才执行，配角不执行
    A = np.array([
        [2., -1., -2.],
        [-4., 6., 3.],
        [-4., -2., 8.]
    ])
    L, U = lu_decompose(A)
    print("原始矩阵 A：\n", A)
    print("\n下三角矩阵 L：\n", L)
    print("\n上三角矩阵 U：\n", U)
    print("\n验证 L @ U：\n", L @ U)
    print("\n是否满足 L @ U ≈ A？", np.allclose(L @ U, A))

原始矩阵 A：
 [[ 2. -1. -2.]
 [-4.  6.  3.]
 [-4. -2.  8.]]

下三角矩阵 L：
 [[ 1.  0.  0.]
 [-2.  1.  0.]
 [-2. -1.  1.]]

上三角矩阵 U：
 [[ 2. -1. -2.]
 [ 0.  4. -1.]
 [ 0.  0.  3.]]

验证 L @ U：
 [[ 2. -1. -2.]
 [-4.  6.  3.]
 [-4. -2.  8.]]

是否满足 L @ U ≈ A？ True


2.已经拿到 L、U 了，解 L U x = b。
    两步（顺序不能反）：
      1) 前代解 L y = b    →  调用 forward_sub
      2) 回代解 U x = y    →  调用 back_sub

In [4]:
import numpy as np
def lu_solve(L, U, b):
    b = np.asarray(b, dtype=float)
    y = forward_sub(L, b)
    x = back_sub(U, y)
    return x
def forward_sub(L, b):
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - np.dot(L[i, :i], y[:i])
    return y
def back_sub(U, y):
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x
import numpy as np
L = np.array([[1, 0, 0], [2, 1, 0], [3, 2, 1]])
b = np.array([7, 21, 41])
def forward_sub(L, b):
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - np.dot(L[i, :i], y[:i])
    return y
y = forward_sub(L, b)
print("前代求得的 y =", y)
def back_sub(U, y):
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):  
        x[i] = (y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x
U = np.array([[2, 1, 1], [0, 2, 1], [0, 0, 2]])
x = back_sub(U, y)
print("回代求得的 x =", x) 

前代求得的 y = [7. 7. 6.]
回代求得的 x = [1. 2. 3.]


3. 一条龙 —— 先 lu_decompose(A) 拿到 L、U，再 lu_solve(L, U, b)。
    这题就是模板卡 `lu_solve` 的"给用户用的那个入口"：
      "同一个 A 换多个 b" 时，用户只需要付一次分解的钱。
    返回：x（长度 n 的一维 numpy 数组）。

In [9]:
def solve_by_lu(A, b):
    L, U = lu_decompose(A)
    x = lu_solve(L, U, b)
    return x
import numpy as np
A = np.array([[2, 1, 1],
              [4, 4, 3],
              [6, 7, 7]])
b = np.array([7, 21, 41])
x = solve_by_lu(A, b)
print("最终解 x =", x)

最终解 x = [1. 2. 3.]


4.带列主元的 LU 分解，返回 (P, L, U)，满足 P @ A ≈ L @ U。
      - P：置换矩阵，每行每列恰好一个 1，其余全 0
           含义：P @ A 就是把 A 的行按"选主元的历史"重排后的矩阵
      - L：单位下三角（对角线全是 1）
      - U：上三角

In [10]:
import numpy as np
def lu_with_pivot(A):
    n = A.shape[0]
    U = A.copy().astype(float)  
    L = np.eye(n)               
    perm = np.arange(n)         
    for k in range(n - 1):
        p = k + np.argmax(np.abs(U[k:, k]))
        if abs(U[p, k]) < 1e-15:
            raise ValueError("矩阵是奇异的，无法进行 LU 分解")
        if p != k:
            U[[k, p], :] = U[[p, k], :]
            L[[k, p], :k] = L[[p, k], :k]
            perm[[k, p]] = perm[[p, k]]
        m = U[k+1:, k] / U[k, k]
        L[k+1:, k] = m
        U[k+1:, k:] -= m[:, np.newaxis] * U[k, k:]
    P = np.zeros((n, n))
    P[np.arange(n), perm] = 1
    return P, L, U
A = np.array([
    [1e-10, 1, 1],
    [2,     3, 4],
    [3,     4, 5]
], dtype=float)
P, L, U = lu_with_pivot(A)
print("P =\n", P)
print("L =\n", L)
print("U =\n", U)
print("P @ A =\n", P @ A)
print("L @ U =\n", L @ U)
print("是否满足 P @ A ≈ L @ U :", np.allclose(P @ A, L @ U))

P =
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]
L =
 [[1.00000000e+00 0.00000000e+00 0.00000000e+00]
 [3.33333333e-11 1.00000000e+00 0.00000000e+00]
 [6.66666667e-01 3.33333333e-01 1.00000000e+00]]
U =
 [[3.         4.         5.        ]
 [0.         1.         1.        ]
 [0.         0.         0.33333333]]
P @ A =
 [[3.e+00 4.e+00 5.e+00]
 [1.e-10 1.e+00 1.e+00]
 [2.e+00 3.e+00 4.e+00]]
L @ U =
 [[3.e+00 4.e+00 5.e+00]
 [1.e-10 1.e+00 1.e+00]
 [2.e+00 3.e+00 4.e+00]]
是否满足 P @ A ≈ L @ U : True


5.组装 n x n 三对角矩阵：
      - 主对角线上的元素全是 main
      - 上、下次对角线上的元素全是 off（两条次对角都是 off）

In [13]:
import numpy as np
def tridiag_matrix(n,main,off):
    main_diag=np.full(n,main)   #n维全是main的数组
    off_diag=np.full(n-1,off)
    matrix=np.diag(main_diag)+np.diag(off_diag,1)+np.diag(off_diag,-1)
    return matrix
n=4
main_val=1
off_val=2
print(tridiag_matrix(n,main_val,off_val))

[[1 2 0 0]
 [2 1 2 0]
 [0 2 1 2]
 [0 0 2 1]]


6.追赶法（Thomas 算法）解三对角方程组 A x = d。要求 O(n)：两遍扫描，
    不许用 np.linalg.solve。参数（注意长度的差别，这是本题第一大坑）：
      a：下对角，长度 n-1，a[i] 在 (i+1, i) 位置
      b：主对角，长度 n
      c：上对角，长度 n-1，c[i] 在 (i, i+1) 位置
      d：右端项，长度 n
    返回：x（长度 n 的一维 numpy 数组）。

In [22]:
import numpy as np
def thomas_solve(a,b,c,d):
    n=len(b)
    if n==1:
        return np.array([d[0]/b[0]])
    c_prime=np.zeros(n)
    d_prime=np.zeros(n)
    c_prime[0]=c[0]/b[0]
    d_prime[0]=d[0]/b[0]
    for i in range(1,n):
        den=b[i]-a[i-1]*c_prime[i-1]
        d_prime[i]=(d[i]-a[i-1]*d_prime[i-1])/den
        if i<n-1:
            c_prime[i]=c[i]/den
    x=np.zeros(n)
    x[n-1]=d_prime[n-1]
    for i in range(n-2,-1,-1):
        x[i]=d_prime[i]-c_prime[i]*x[i+1]
    return x
a=np.array([-2,-3,-4])
b=np.array([4,5,6,7])
c=np.array([-1,-1,-1])
d=np.array([2,5,8,16])
print(thomas_solve(a,b,c,d))

[1. 2. 3. 4.]


7."看到三对角就写这两步"的完整流程：
      1) 组装：A = tridiag_matrix(n, main, off)
      2) 求解：x = thomas_solve(a, b, c, d)

8.用已经分解好的 (L, U) 求行列式。
    公式：det(A) = det(L) * det(U)；三角矩阵的行列式 = 对角线元素的乘积；
         而 L 是单位下三角，det(L) = 1。

In [23]:
import numpy as np
def det_from_lu(L,U):
    return float(np.prod(np.diag(U)))  #prod为连乘
L=np.array([[1,0,0],
            [2,1,0],
            [3,4,1]])
U=np.array([[2,3,4],
            [0,5,6],
            [0,0,10]])
print(det_from_lu(L,U))

100.0


9.无穷范数条件数
        cond_inf(A) = ||A||_inf * ||A^{-1}||_inf
    其中无穷范数 = 【每一行绝对值之和】的最大值，即 max_i Σ_j |A[i][j]|。

In [24]:
import numpy as np

def condition_number_inf(A):
    """
    第 11 题：无穷范数条件数
    """
    return float(np.linalg.cond(A, np.inf))

# 构造一个测试矩阵
A = np.array([
    [1.0, 2.0],
    [3.0, 4.0]
])

print(condition_number_inf(A))

20.999999999999993


10.相对误差放大倍数 —— 亲眼看到"条件数决定误差放大"。

In [25]:
import numpy as np

def error_amplification(A, b, db):
    if np.all(db == 0):
        return 0.0
    x = np.linalg.solve(A, b)
    xp = np.linalg.solve(A, b + db)
    dx = xp - x
    norm_x = np.linalg.norm(x, np.inf)
    norm_b = np.linalg.norm(b, np.inf)
    norm_dx = np.linalg.norm(dx, np.inf)
    norm_db = np.linalg.norm(db, np.inf)
    return float((norm_dx / norm_x) / (norm_db / norm_b))

# 1. 良态矩阵 (条件数较小，误差放大不明显)
A_good = np.array([[2.0, 0.0], 
                   [0.0, 2.0]])
b_good = np.array([2.0, 2.0])
db_good = np.array([0.02, 0.0]) # 微小扰动

print("良态矩阵放大倍数:", error_amplification(A_good, b_good, db_good))

# 2. 病态矩阵 (Hilbert 矩阵的简化版，条件数较大，误差可能被恐怖放大)
A_bad = np.array([[1.0, 2.0], 
                  [2.0, 4.001]]) # 几乎奇异
b_bad = np.array([3.0, 5.0])
db_bad = np.array([0.001, 0.0]) # 微小扰动

print("病态矩阵放大倍数:", error_amplification(A_bad, b_bad, db_bad))

# 3. 测试 db 全为 0 的情况
db_zero = np.array([0.0, 0.0])
print("db 全为 0 时:", error_amplification(A_good, b_good, db_zero))

良态矩阵放大倍数: 1.0000000000000009
病态矩阵放大倍数: 9.987518721915853
db 全为 0 时: 0.0
